# Tracking de partículas en movimiento browniano

Este notebook obtiene las trayectorias (_tracks_) de las partículas libres de un video de microscopía.
Lo usamos para las partículas de látex en glicerina y para las gotas de grasa de crema.

El procedimiento tiene estos pasos:

1. Dividir cada frame por el fondo, para eliminar la iluminación no uniforme.
2. Detectar las partículas en cada frame.
3. Calcular el centro de cada partícula con precisión subpíxel (simetría radial).
4. Descartar las detecciones alargadas o poco simétricas.
5. Unir las detecciones de frames sucesivos en tracks.
6. Cortar los tracks donde la unión es ambigua.
7. Descartar los tracks cortos y los de partículas pegadas al vidrio.
8. (Solo látex) Medir el diámetro aparente de cada track, para separar las partículas de látex de las manchas.

El resultado es un archivo `tracks.csv` con las columnas `frame, x, y, track_id` (posiciones en píxeles).

**Nota:** es una versión simplificada del script `track.py` que usamos en el informe.
Los umbrales de detección están fijados con los valores que obtuvimos al calibrar con el video de fondo.

In [ ]:
from collections import deque
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import trackpy as tp
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.spatial import cKDTree

tp.quiet()

## Parámetros

Elegimos el video y la muestra. Cada muestra tiene sus propios parámetros:

- `search_range`: distancia máxima (px) entre la posición prevista de una partícula y su detección en el frame siguiente.
- `memory`: cantidad de frames que una partícula puede desaparecer sin que se corte su track.
- `max_sym_err`: cuánto puede apartarse una detección de la simetría radial (0 = perfectamente redonda).
- `min_track_len`: largo mínimo de un track, en frames.

Los umbrales de detección (`minmass`, `obj_threshold`, `peak_threshold`) separan las partículas del ruido.
Los obtuvimos de las detecciones falsas en el video de fondo, que no tiene partículas.

In [ ]:
MUESTRA = 'latex_glycerin'        # 'latex_glycerin' o 'water_cream'
VIDEO = 'latex_glycerin_1'

CARPETA = Path('videos') / MUESTRA      # videos/<muestra>/<video>.avi y el video de fondo
SALIDA = Path('resultados') / VIDEO
SALIDA.mkdir(parents=True, exist_ok=True)

CONFIGS = {
    'latex_glycerin': dict(
        fondo='latex_glycerin_background.avi',
        detector='trackpy',
        diameter=41,             # px: las partículas se ven como discos oscuros de ~40 px
        separation=20,           # px: dos detecciones más cercanas se consideran una sola
        noise_size=2.0,
        minmass=8155.0,          # umbral de detección (calibrado con el fondo)
        strict_min_mass=2.0,     # una detección "estricta" necesita al menos 2 * minmass
        crowd_dist=50.0,         # px: con un vecino más cerca, la detección nunca es estricta
        rs_radius=22,            # px: radio de la ventana de simetría radial
        grad_sigma=2.0,
        max_sym_err=0.30, loose_sym_err=0.35,
        search_range=8.0, adaptive_stop=2.0, memory=5,
        min_track_len=20,
        static_window=0,         # sin mediana temporal: las partículas se mueven poco y la mediana las borraría
    ),
    'water_cream': dict(
        fondo='water_cream_background.avi',
        detector='objetos',
        obj_sigma=3.0,           # suavizado del mapa de objetos
        obj_threshold=18.5,      # umbral para la extensión de un objeto (calibrado con el fondo)
        peak_threshold=32.8,     # umbral para el pico de un objeto (calibrado con el fondo)
        obj_merge=3,             # px: une manchas vecinas en un mismo objeto
        min_obj_area=30,         # px
        max_obj_diam=60.0,       # px
        max_axis_ratio=2.0,      # objetos más alargados se descartan (pares pegados)
        grad_sigma=1.5,
        max_sym_err=0.35, loose_sym_err=0.5,
        search_range=15.0, adaptive_stop=3.0, memory=5,
        min_track_len=20,
        static_window=75,        # +- frames de la mediana temporal que elimina lo que no se mueve
    ),
}
P = CONFIGS[MUESTRA]
P.update(vel_history=10, vel_min_span=4, min_strict_frac=0.5, static_lag=50, static_ratio=0.5)

## Paso 1: dividir por el fondo

Antes de colocar la muestra filmamos un video de fondo.
Tomamos la mediana de sus frames y dividimos cada frame del video por esa imagen.
Así eliminamos la iluminación no uniforme y las manchas fijas de la óptica.

Luego restamos 1 y multiplicamos por 1000: la imagen resultante vale cero donde no hay nada.

In [ ]:
def leer_frames(path, cada=1):
    """Devuelve (índice, frame en escala de grises) para cada frame del video."""
    cap = cv2.VideoCapture(str(path))
    i = 0
    while True:
        ok, f = cap.read()
        if not ok:
            break
        if i % cada == 0:
            yield i, cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
        i += 1
    cap.release()


def cociente(gris, fondo):
    r = gris.astype(np.float32) / fondo
    return r / np.median(r)


fondo = np.median([g.astype(np.float32) for _, g in leer_frames(CARPETA / P['fondo'])], axis=0)

video = CARPETA / f'{VIDEO}.avi'
cap = cv2.VideoCapture(str(video))
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f'{n_frames} frames a {fps:.2f} fps')

### Mediana temporal (solo crema)

En la crema quedan manchas fijas que no están en el video de fondo, y partículas pegadas al vidrio.
Para eliminarlas, restamos la mediana de los frames vecinos (± 75 frames): lo que no se mueve desaparece.
En el látex no la usamos, porque las partículas se mueven tan poco que la mediana también las borraría.

In [ ]:
centros, estaticas = [0], [np.ones_like(fondo)]
if P['static_window']:
    sub = dict(leer_frames(video, cada=5))
    idx = np.array(sorted(sub))
    centros = np.arange(0, n_frames, 25)          # recalculamos la mediana cada 25 frames
    estaticas = [np.median([cociente(sub[i], fondo) for i in idx[np.abs(idx - c) <= P['static_window']]], axis=0)
                 for c in centros]
centros = np.array(centros)


def imagen_limpia(gris, i):
    """Frame dividido por el fondo, menos la parte estática, en milésimas."""
    k = np.argmin(np.abs(centros - i))
    return (cociente(gris, fondo) - estaticas[k]) * 1000

## Paso 2: centro por simetría radial

Para ubicar el centro de cada partícula usamos el método de simetría radial (Parthasarathy, 2012).
En una partícula redonda, todos los gradientes de intensidad apuntan hacia el centro.
El método busca el punto más cercano a todas las rectas de los gradientes.

Además devuelve un error: la distancia media de esas rectas al centro.
Es cero para una partícula perfectamente redonda y crece para pares pegados o suciedad.
Ese error es el que usamos para descartar detecciones.

In [ ]:
def centro_simetria_radial(I, pesos=None):
    """Centro (x, y) de la ventana I y error de simetría (px)."""
    Ny, Nx = I.shape
    xm, ym = np.meshgrid(np.arange(Nx - 1) - (Nx - 2) / 2, np.arange(Ny - 1) - (Ny - 2) / 2)
    # gradientes en las dos diagonales, suavizados
    du = ndi.uniform_filter(I[:-1, 1:] - I[1:, :-1], 3, mode='nearest')
    dv = ndi.uniform_filter(I[:-1, :-1] - I[1:, 1:], 3, mode='nearest')
    g2 = du**2 + dv**2
    if pesos is not None:
        g2 = g2 * pesos
    with np.errstate(divide='ignore', invalid='ignore'):
        m = -(dv + du) / (du - dv)                  # pendiente de la recta de cada gradiente
    malo = ~np.isfinite(m)
    m[malo], g2[malo] = 0, 0
    if g2.sum() <= 0:
        return np.nan, np.nan, np.inf
    b = ym - m * xm
    # mínimos cuadrados pesados: punto más cercano a todas las rectas
    x0, y0 = (g2 * xm).sum() / g2.sum(), (g2 * ym).sum() / g2.sum()
    w = g2 / np.sqrt((xm - x0)**2 + (ym - y0)**2 + 1e-9) / (m**2 + 1)
    sw, smmw, smw = w.sum(), (m * m * w).sum(), (m * w).sum()
    smbw, sbw = (m * b * w).sum(), (b * w).sum()
    det = smw**2 - smmw * sw
    xc = (smbw * sw - smw * sbw) / det
    yc = (smbw * smw - smmw * sbw) / det
    error = (g2 * np.abs(m * xc - yc + b) / np.sqrt(m**2 + 1)).sum() / g2.sum()
    return xc + (Nx - 1) / 2, yc + (Ny - 1) / 2, error

## Paso 3: detección

Usamos un detector distinto para cada muestra.

**Látex.** Las partículas son discos oscuros regulares.
Invertimos la imagen y usamos `trackpy.locate`, que encuentra manchas brillantes de un diámetro dado.
Después refinamos el centro con simetría radial.

**Crema.** Las gotas desenfocadas se ven como anillos, y `trackpy` las parte en varias detecciones.
Por eso buscamos objetos: zonas donde el valor absoluto de la imagen (suavizada) supera un umbral.
Cada objeto es una detección, y su centro también sale de la simetría radial.

Cada detección queda marcada de dos formas:

- `keep`: suficientemente redonda para formar parte de un track.
- `strict`: muy redonda y aislada. Un track debe empezar y terminar en detecciones estrictas (Paso 5).

In [ ]:
def detectar_latex(i, gris):
    d = imagen_limpia(gris, i)
    f = tp.locate(-d, P['diameter'], minmass=P['minmass'], separation=P['separation'],
                  noise_size=P['noise_size'])
    f = f.sort_values('mass', ascending=False)
    ds = cv2.GaussianBlur(d, (0, 0), P['grad_sigma']).astype(np.float64)
    H, W = d.shape
    R = P['rs_radius']
    yy, xx = np.mgrid[0:2 * R, 0:2 * R]
    disco = (np.hypot(xx - R + 0.5, yy - R + 0.5) <= R).astype(float)

    filas, tomados = [], []
    for _, p in f.iterrows():
        x, y, err = p.x, p.y, np.inf
        for _ in range(2):                          # dos iteraciones de simetría radial
            x0, y0 = int(round(x)), int(round(y))
            if x0 - R < 0 or y0 - R < 0 or x0 + R + 1 > W or y0 + R + 1 > H:
                x = np.nan                          # demasiado cerca del borde
                break
            xr, yr, err = centro_simetria_radial(ds[y0 - R:y0 + R + 1, x0 - R:x0 + R + 1], disco)
            x, y = xr + x0 - R, yr + y0 - R
        if not np.isfinite(x) or any(np.hypot(x - a, y - b) < 20 for a, b in tomados):
            continue                                # fuera del cuadro, o la misma partícula dos veces
        tomados.append((x, y))
        sym = err / R
        ok = np.hypot(x - p.x, y - p.y) <= R        # el centro no se fue lejos de la detección
        filas.append(dict(frame=i, x=x, y=y, mass=p.mass, size=p['size'],
                          keep=ok and sym <= P['loose_sym_err'],
                          strict=ok and sym <= P['max_sym_err'] and p.mass >= P['strict_min_mass'] * P['minmass']))
    out = pd.DataFrame(filas)
    # una partícula con otra muy cerca (par pegado) nunca es estricta
    if len(out) > 1:
        dist, _ = cKDTree(out[['x', 'y']].values).query(out[['x', 'y']].values, k=2)
        out['strict'] &= dist[:, 1] >= P['crowd_dist']
    return out

In [ ]:
def detectar_crema(i, gris):
    d = imagen_limpia(gris, i)
    H, W = d.shape
    E = cv2.GaussianBlur(np.abs(d), (0, 0), P['obj_sigma'])     # mapa de objetos
    fg = E > P['obj_threshold']
    etiquetas, n = ndi.label(ndi.binary_dilation(fg, iterations=P['obj_merge']))
    etiquetas = etiquetas * fg
    ds = cv2.GaussianBlur(d, (0, 0), P['grad_sigma']).astype(np.float64)

    filas = []
    for k, sl in enumerate(ndi.find_objects(etiquetas)):
        if sl is None:
            continue
        obj = etiquetas[sl] == k + 1
        area = obj.sum()
        if area < P['min_obj_area'] or E[sl][obj].max() < P['peak_threshold']:
            continue                                # muy chico o muy débil: ruido
        yy, xx = np.nonzero(obj)
        w = E[sl][obj] - P['obj_threshold']
        cx, cy = (xx * w).sum() / w.sum() + sl[1].start, (yy * w).sum() / w.sum() + sl[0].start
        # alargamiento: cociente de los ejes de la mancha
        ev = np.sort(np.linalg.eigvalsh(np.cov(np.vstack([xx, yy]), aweights=w)))
        alargamiento = np.sqrt(ev[1] / max(ev[0], 1e-6))
        radio = np.sqrt(area / np.pi)
        # simetría radial en una ventana alrededor del objeto
        R = int(np.clip(radio + 4, 7, 30))
        x0, y0 = int(round(cx)), int(round(cy))
        ya, yb, xa, xb = max(y0 - R, 0), min(y0 + R + 1, H), max(x0 - R, 0), min(x0 + R + 1, W)
        mascara = ndi.binary_dilation(etiquetas[ya:yb, xa:xb] == k + 1, iterations=3)[:-1, :-1]
        x, y, err = centro_simetria_radial(ds[ya:yb, xa:xb], mascara)
        x, y = x + xa, y + ya
        sym = err / radio
        ok = (np.isfinite(x) and min(x, y, W - 1 - x, H - 1 - y) >= radio      # no toca el borde
              and 2 * radio <= P['max_obj_diam'] and alargamiento <= P['max_axis_ratio']
              and np.hypot(x - cx, y - cy) <= radio)
        filas.append(dict(frame=i, x=x if np.isfinite(x) else cx, y=y if np.isfinite(y) else cy,
                          mass=np.abs(d[sl][obj]).sum(), size=radio,
                          keep=ok and sym <= P['loose_sym_err'], strict=ok and sym <= P['max_sym_err']))
    return pd.DataFrame(filas)

Corremos la detección en todos los frames.
Guardamos todas las detecciones, también las descartadas, porque las usamos en el Paso 6 para encontrar uniones ambiguas.

Este paso es el más lento (unos minutos para el video 1 de látex, cerca de una hora para el video 2).

In [ ]:
detectar = detectar_latex if P['detector'] == 'trackpy' else detectar_crema
todas = pd.concat([detectar(i, g) for i, g in leer_frames(video)], ignore_index=True)
todas.to_pickle(SALIDA / 'detecciones.pkl')
print(f'{len(todas)} detecciones, {todas.keep.sum()} utilizables, {todas.strict.sum()} estrictas')

## Paso 4: unión de detecciones en tracks

Para unir las detecciones usamos `trackpy`.
En cada frame, `trackpy` busca cada partícula cerca de su posición prevista, a menos de `search_range` píxeles.

La posición prevista sale de la velocidad media de la partícula en sus últimos 10 frames.
Así seguimos bien a las partículas que derivan con el fluido.
Si la partícula todavía tiene poca historia, usamos la velocidad mediana de todas las partículas (la deriva global).

In [ ]:
class PrediccionVelocidad(tp.predict.NullPredict):
    """Prevé la posición de cada partícula con su velocidad reciente, o con la deriva global."""

    def __init__(self):
        self.pos_columns = ['x', 'y']
        self.historia = {}           # partícula -> últimas posiciones (t, x, y)
        self.quien = {}              # t -> {(x, y): partícula}
        self.v_global = np.zeros(2)
        self.v_global_por_frame = {}

    def velocidad(self, h):
        if h[-1][0] - h[0][0] >= P['vel_min_span']:
            return np.array([h[-1][1] - h[0][1], h[-1][2] - h[0][2]]) / (h[-1][0] - h[0][0])
        return None

    def observe(self, frame):
        t = int(frame[self.t_column].iloc[0])
        self.quien[t] = {}
        vs = []
        for pid, x, y in zip(frame.particle, frame.x, frame.y):
            h = self.historia.setdefault(pid, deque(maxlen=P['vel_history'] + 1))
            h.append((t, x, y))
            self.quien[t][(x, y)] = pid
            v = self.velocidad(h)
            if v is not None:
                vs.append(v)
        if vs:
            self.v_global = np.median(vs, axis=0)
        self.v_global_por_frame[t] = self.v_global.copy()

    def predict(self, t1, particulas):
        out = []
        for p in particulas:
            pid = self.quien.get(p.t, {}).get(tuple(p.pos))
            v = self.velocidad(self.historia[pid]) if pid is not None else None
            out.append(np.asarray(p.pos) + (self.v_global if v is None else v) * (t1 - p.t))
        return np.array(out)


pred = PrediccionVelocidad()
utiles = todas[todas.keep].reset_index(drop=True)
t = pred.link_df(utiles, P['search_range'], memory=P['memory'],
                 adaptive_stop=P['adaptive_stop'], adaptive_step=0.95)
v_global = pred.v_global_por_frame
print(f'{t.particle.nunique()} tracks iniciales')

## Paso 5: cortar las uniones ambiguas

Si cerca de la posición prevista había dos detecciones, no sabemos cuál es la correcta.
En ese caso no elegimos: cortamos el track en ese frame.
Contamos como candidatas todas las detecciones, también las descartadas por forma.

Preferimos perder datos antes que unir mal, porque una unión equivocada agrega un salto falso que aumenta el MSD.

In [ ]:
t = t.sort_values(['particle', 'frame']).reset_index(drop=True)
arboles = {fr: cKDTree(d[['x', 'y']].values) for fr, d in todas.groupby('frame')}
fr, xy, pid = t.frame.values, t[['x', 'y']].values, t.particle.values

corte = np.zeros(len(t), bool)
inicio = 0
for k in range(1, len(t)):
    if pid[k] != pid[k - 1]:
        inicio = k
        continue
    a, b = k - 1, k
    # la misma predicción que usó trackpy
    j0 = max(inicio, a - P['vel_history'])
    if fr[a] - fr[j0] >= P['vel_min_span']:
        v = (xy[a] - xy[j0]) / (fr[a] - fr[j0])
    else:
        v = v_global.get(fr[b] - 1, np.zeros(2))
    dt = fr[b] - fr[a]
    n_destino = len(arboles[fr[b]].query_ball_point(xy[a] + v * dt, P['search_range']))
    n_origen = len(arboles[fr[a]].query_ball_point(xy[b] - v * dt, P['search_range']))
    corte[b] = n_destino > 1 or n_origen > 1

t['track_id'] = np.cumsum(corte | np.r_[True, pid[1:] != pid[:-1]])
print(f'{corte.sum()} uniones ambiguas cortadas')

## Paso 6: limpiar los tracks

Aplicamos tres filtros:

1. **Detecciones estrictas.** Recortamos cada track para que empiece y termine en detecciones estrictas, y pedimos que al menos la mitad lo sean. Las detecciones menos redondas solo sirven para continuar un track ya empezado.
2. **Partículas pegadas.** Una partícula libre tiene un MSD que crece con el lag. Si el MSD a un lag largo es mucho menor de lo esperado, la partícula está pegada al vidrio y la descartamos.
3. **Tracks cortos.** Descartamos los tracks de menos de 20 frames.

In [ ]:
def recortar_estrictas(g):
    e = g.strict.values
    if not e.any():
        return None
    g = g.iloc[np.argmax(e):len(e) - np.argmax(e[::-1])]
    return g if g.strict.mean() >= P['min_strict_frac'] else None


def esta_pegada(g):
    """MSD(lag largo) mucho menor que lag * MSD(1 frame): la partícula no difunde libremente."""
    fr = g.frame.values
    pos = dict(zip(fr, g[['x', 'y']].values))
    lag = int(min(P['static_lag'], (fr[-1] - fr[0]) // 2))
    d1 = [np.sum((pos[f + 1] - pos[f])**2) for f in fr if f + 1 in pos]
    dl = [np.sum((pos[f + lag] - pos[f])**2) for f in fr if f + lag in pos]
    if len(d1) < 5 or len(dl) < 3:
        return False
    return np.mean(dl) < P['static_ratio'] * lag * np.mean(d1)


t = pd.concat([g for g in (recortar_estrictas(g) for _, g in t.groupby('track_id')) if g is not None])
pegadas = [tid for tid, g in t.groupby('track_id') if esta_pegada(g)]
t = t[~t.track_id.isin(pegadas)]
largo = t.groupby('track_id').frame.transform(lambda s: s.max() - s.min() + 1)
t = t[largo >= P['min_track_len']].copy()
t['track_id'] = pd.factorize(t.track_id)[0]     # renumeramos desde 0

t[['frame', 'x', 'y', 'mass', 'size', 'track_id']].to_csv(SALIDA / 'tracks.csv', index=False)
print(f'{t.track_id.nunique()} tracks finales ({len(pegadas)} pegadas descartadas), '
      f'{len(t)} posiciones, {len(t) / n_frames:.1f} partículas por frame')

## Verificación visual

Dibujamos todos los tracks sobre un frame del video.
Es la forma más directa de comprobar que el tracking sigue a las partículas.
Para revisarlo en detalle conviene hacer un video con los tracks superpuestos (celda siguiente).

In [ ]:
cap = cv2.VideoCapture(str(video))
cap.set(cv2.CAP_PROP_POS_FRAMES, n_frames // 2)
_, img = cap.read()
cap.release()

fig, ax = plt.subplots(figsize=(8, 6.4))
ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cmap='gray')
for _, g in t.groupby('track_id'):
    ax.plot(g.x, g.y, lw=0.8)
ax.set_title(f'{VIDEO}: {t.track_id.nunique()} tracks')
ax.axis('off')
plt.show()

In [ ]:
def video_superpuesto(path_salida, cola=20):
    """Video con un círculo, el número y la cola de cada track."""
    cap = cv2.VideoCapture(str(video))
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(str(path_salida), cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
    por_frame = dict(list(t.groupby('frame')))
    i = 0
    while True:
        ok, img = cap.read()
        if not ok:
            break
        for _, p in por_frame.get(i, pd.DataFrame()).iterrows():
            g = t[(t.track_id == p.track_id) & (t.frame <= i) & (t.frame > i - cola)]
            cv2.polylines(img, [g[['x', 'y']].values.astype(np.int32)], False, (255, 0, 255), 1)
            cv2.circle(img, (int(p.x), int(p.y)), 10, (255, 0, 255), 1)
            cv2.putText(img, str(int(p.track_id)), (int(p.x) + 10, int(p.y) - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 255), 1)
        out.write(img)
        i += 1
    cap.release()
    out.release()


# video_superpuesto(SALIDA / 'tracks.mp4')     # tarda varios minutos

## Paso 7: diámetro aparente (solo látex)

En la muestra de látex aparecen, además de las partículas, manchas oscuras más chicas que también se mueven.
Para separarlas medimos el diámetro aparente de cada track:

1. Recortamos una ventana de 81 × 81 px centrada en la partícula, en 12 frames repartidos a lo largo del track.
2. Promediamos esos recortes. El promedio reduce el ruido y deja una imagen nítida de la partícula.
3. Calculamos el perfil radial: la intensidad media en anillos de 1 px alrededor del centro.
4. El diámetro aparente es el doble del radio en el que lo oscuro cae a la mitad de su máximo.

Una partícula de 3,24 µm no puede verse menor que su tamaño real.
Por eso descartamos los tracks con diámetro aparente menor que 3,0 µm.

In [ ]:
UM_POR_PX = 0.0948
H = 40                                          # media ventana (px)
CORTE = 3.0                                     # µm

# qué frames necesitamos de cada track
pedidos = {}
for tid, g in t.groupby('track_id'):
    filas = g.iloc[np.linspace(0, len(g) - 1, min(12, len(g))).astype(int)]
    for _, r in filas.iterrows():
        pedidos.setdefault(int(r.frame), []).append((tid, r.x, r.y))

recortes = {tid: [] for tid in t.track_id.unique()}
for i, gris in leer_frames(video):
    if i not in pedidos:
        continue
    img = np.pad(cociente(gris, fondo), H + 2, mode='edge')
    for tid, x, y in pedidos[i]:
        # desplazamiento subpíxel para centrar la partícula en la ventana
        M = np.float32([[1, 0, -(x + 2)], [0, 1, -(y + 2)]])
        recortes[tid].append(cv2.warpAffine(img, M, (2 * H + 1, 2 * H + 1), flags=cv2.INTER_LINEAR))
promedios = {tid: np.mean(c, axis=0) for tid, c in recortes.items()}

In [ ]:
yy, xx = np.mgrid[-H:H + 1, -H:H + 1]
r = np.hypot(xx, yy)


def diametro_aparente(c):
    perfil = np.array([c[(r >= k) & (r < k + 1)].mean() for k in range(H)])
    oscuro = perfil[32:].mean() - perfil         # cuánto más oscuro que el fondo lejano
    k = 4 + np.where(oscuro[4:] >= oscuro[4:].max() / 2)[0].max()   # último radio sobre la mitad
    return 2 * (k + 0.5) * UM_POR_PX


sel = pd.DataFrame({'track_id': list(promedios),
                    'diametro_um': [diametro_aparente(c) for c in promedios.values()]})
sel['latex'] = sel.diametro_um >= CORTE
sel.to_csv(SALIDA / 'seleccion_latex.csv', index=False)
print(f'{sel.latex.sum()} partículas de látex y {(~sel.latex).sum()} manchas')

plt.hist([sel.diametro_um[~sel.latex], sel.diametro_um[sel.latex]], bins=20, stacked=True,
         label=['manchas', 'látex'])
plt.axvline(CORTE, color='k', ls='--')
plt.xlabel('diámetro aparente [µm]')
plt.ylabel('número de tracks')
plt.legend()
plt.show()